# Test / benchmark a trained checkpoint on Colab

Loads a checkpoint (e.g. `best.pt` produced by `colab_train_vimeo.ipynb`) from Drive, and measures every metric this project currently implements:

- **Raw reconstruction quality** (MSE / PSNR, no quantization, no entropy coding) plus a visual Original | Reconstruction comparison
- **Full codec benchmark** at several bit depths (8/6/4-bit by default): PSNR, bits-per-pixel, compression ratio vs. raw RGB, entropy-coding efficiency against the theoretical model, quantization clipping, and encode/decode timing per frame
- Losslessness of the entropy-coding stage is verified as part of the benchmark itself, not assumed

**By default this needs nothing from you**: it auto-downloads DAVIS 2017 TrainVal 480p (~795 MB, official host, verified reachable) each session and evaluates against that. If your checkpoint was trained on Vimeo-90K, this is a genuinely clean test - the model has never seen a DAVIS frame, from either DAVIS's train or test split, so there's no memorization risk in either direction.

**To test a different dataset later:** set `USE_BUILTIN_DAVIS = False` and fill in `DATASET_INPUT_PATH`/`DATASET_LABEL` in the config cell - nothing else needs to change. Any folder of video files, or a folder whose subfolders are each an image sequence, works via the same auto-detecting ingestion this project already uses for DAVIS.

**Everything here reuses the project's own tested scripts** (`prepare_dataset.py`, `reconstruct.py`, `calibrate_quantizer.py`, `benchmark_codec.py`) rather than reimplementing the benchmark math - results are produced the same way, by the same code, as every other measured number in this project.

**Results are saved to Drive**, namespaced by `DATASET_LABEL`, so testing multiple datasets over time doesn't overwrite previous results.

In [ ]:
from pathlib import Path

# --- The trained model to test ---
CHECKPOINT_PATH = Path('/content/drive/MyDrive/neural_streaming_colab/checkpoints/best.pt')

# --- The test dataset ---
# Default: auto-downloads DAVIS 2017 TrainVal 480p (~795 MB, official host) fresh
# each session - nothing to upload yourself. This is a clean test for a
# Vimeo-trained checkpoint: the model has never seen a single DAVIS frame.
USE_BUILTIN_DAVIS = True

# To test a different dataset instead: set USE_BUILTIN_DAVIS = False and fill
# these in yourself. DATASET_INPUT_PATH can be a Drive folder, or anywhere else
# this Colab session can read from (e.g. something you download in an earlier
# cell, the same way the DAVIS auto-download below works).
DATASET_LABEL = 'davis'  # short name, used to namespace outputs - no spaces
DATASET_INPUT_PATH = Path('/content/drive/MyDrive/datasets/your_dataset_here')
DATASET_SOURCE_TYPE = None  # None = auto-detect ('video' vs 'image-sequence'), or force one explicitly

# --- Frame preparation (must match what the checkpoint was trained on - divisible by 16) ---
FRAME_WIDTH = 256
FRAME_HEIGHT = 256
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

# --- Codec benchmark ---
BIT_WIDTHS = [8, 6, 4]
CALIBRATION_MAX_BATCHES = 50  # training batches used to calibrate quantization + entropy model
BATCH_SIZE = 8

# --- Where things live ---
REPO_URL = 'https://github.com/yuvidewan/neural_streaming.git'
REPO_DIR = Path('/content/neural_streaming')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/neural_streaming_colab/test_results') / DATASET_LABEL

## 1. Mount Google Drive

The checkpoint is read from here, and results are written back here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results for dataset {DATASET_LABEL!r} will be saved to: {DRIVE_RESULTS_DIR}')

## 2. Get the project code

In [ ]:
import os
import subprocess

if REPO_DIR.is_dir():
    print(f'{REPO_DIR} already exists - pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt
!pip install -e .

In [ ]:
# Fails loudly right here if the editable install above didn't actually take -
# better than a confusing ModuleNotFoundError several cells later. If this
# fails right after a fresh install, Restart session and Run all once more -
# editable-install path registration only takes effect on the NEXT interpreter
# startup, not the one that ran the install.
import nvc
print(f'nvc package loaded OK from: {nvc.__file__}')

## 3. Verify the checkpoint

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. '
        'Make sure Drive is mounted and this points at a real best.pt / latest.pt.'
    )
size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
print(f'Using checkpoint: {CHECKPOINT_PATH} ({size_mb:.1f} MB)')

## 4. Get the test dataset

If `USE_BUILTIN_DAVIS` is `True` (the default), this downloads and extracts DAVIS 2017 TrainVal 480p directly from the official host - verified reachable, ~795 MB - and points `DATASET_INPUT_PATH` at it automatically. No upload, no manual Drive step. If you've set `USE_BUILTIN_DAVIS = False`, this cell does nothing and your own `DATASET_INPUT_PATH` is used as-is.

In [ ]:
import urllib.request
import zipfile

DAVIS_URL = 'https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip'
DAVIS_DOWNLOAD_DIR = Path('/content/davis_download')

if USE_BUILTIN_DAVIS:
    zip_path = DAVIS_DOWNLOAD_DIR / 'DAVIS-2017-trainval-480p.zip'
    extract_dir = DAVIS_DOWNLOAD_DIR / 'extracted'
    davis_root = extract_dir / 'DAVIS' / 'JPEGImages' / '480p'

    if davis_root.is_dir():
        print(f'DAVIS already downloaded and extracted - reusing {davis_root}')
    else:
        DAVIS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
        print('Downloading DAVIS 2017 TrainVal 480p (~795 MB) from the official host...')
        urllib.request.urlretrieve(DAVIS_URL, zip_path)

        print('Extracting...')
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        zip_path.unlink()

    DATASET_INPUT_PATH = davis_root
    DATASET_SOURCE_TYPE = 'image-sequence'
    print(f'DATASET_INPUT_PATH set to: {DATASET_INPUT_PATH}')
else:
    print(f'USE_BUILTIN_DAVIS is False - using DATASET_INPUT_PATH as configured: {DATASET_INPUT_PATH}')

## 5. Prepare the test dataset

Reuses `scripts/prepare_dataset.py` exactly as documented in the main README - it auto-detects whether `DATASET_INPUT_PATH` is a folder of videos or a folder of image-sequence subfolders (DAVIS's layout), resizes/crops every frame to `FRAME_WIDTH` x `FRAME_HEIGHT`, and writes a leakage-safe train/val/test split. Everything after this cell is dataset-agnostic.

In [ ]:
LOCAL_FRAMES_DIR = Path('/content/test_frames') / DATASET_LABEL
LOCAL_MANIFEST_PATH = Path('/content/test_manifest') / f'{DATASET_LABEL}_manifest.json'

if not DATASET_INPUT_PATH.is_dir():
    raise FileNotFoundError(
        f'DATASET_INPUT_PATH does not exist: {DATASET_INPUT_PATH}. '
        'Point it at a folder of videos, or a folder whose subfolders are each an image sequence.'
    )

cmd = [
    'python', 'scripts/prepare_dataset.py',
    '--input', str(DATASET_INPUT_PATH),
    '--output', str(LOCAL_FRAMES_DIR),
    '--manifest', str(LOCAL_MANIFEST_PATH),
    '--width', str(FRAME_WIDTH),
    '--height', str(FRAME_HEIGHT),
    '--train-ratio', str(TRAIN_RATIO),
    '--val-ratio', str(VAL_RATIO),
    '--test-ratio', str(TEST_RATIO),
    '--seed', str(SEED),
]
if DATASET_SOURCE_TYPE is not None:
    cmd += ['--source-type', DATASET_SOURCE_TYPE]

subprocess.run(cmd, check=True)

## 6. Reconstruction quality (no quantization) + visual comparison

Reuses `scripts/reconstruct.py`: encodes/decodes a sample of test-split frames through the raw autoencoder (no quantization or entropy coding yet), reports MSE/PSNR, and saves an Original | Reconstruction comparison grid.

In [ ]:
import shutil

RECON_OUTPUT = Path('/content/reconstruction.png')

subprocess.run([
    'python', 'scripts/reconstruct.py',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--manifest', str(LOCAL_MANIFEST_PATH),
    '--num-samples', '8',
    '--output', str(RECON_OUTPUT),
], check=True)

shutil.copy(RECON_OUTPUT, DRIVE_RESULTS_DIR / 'reconstruction.png')

from IPython.display import Image, display
display(Image(filename=str(RECON_OUTPUT)))

## 7. Full codec benchmark (quantization + entropy coding)

Reuses `scripts/calibrate_quantizer.py` (once per bit width, calibrating fixed quantization parameters and an entropy model from this dataset's own TRAIN split only - never val/test) and `scripts/benchmark_codec.py` (which then encodes and decodes every TEST-split frame through the real `.nvc` path and reports measured PSNR, BPP, compression ratio, and entropy-coding efficiency).

In [ ]:
CALIBRATION_DIR = Path('/content/calibration') / DATASET_LABEL
CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)

calibration_paths = []
for bits in BIT_WIDTHS:
    output_path = CALIBRATION_DIR / f'calibration_{bits}bit.json'
    subprocess.run([
        'python', 'scripts/calibrate_quantizer.py',
        '--checkpoint', str(CHECKPOINT_PATH),
        '--manifest', str(LOCAL_MANIFEST_PATH),
        '--bits', str(bits),
        '--batch-size', str(BATCH_SIZE),
        '--max-batches', str(CALIBRATION_MAX_BATCHES),
        '--seed', str(SEED),
        '--output', str(output_path),
    ], check=True)
    calibration_paths.append(output_path)

In [ ]:
BENCHMARK_METRICS_DIR = Path('/content/metrics') / DATASET_LABEL

benchmark_cmd = [
    'python', 'scripts/benchmark_codec.py',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--calibration', *[str(p) for p in calibration_paths],
    '--manifest', str(LOCAL_MANIFEST_PATH),
    '--seed', str(SEED),
    '--metrics-dir', str(BENCHMARK_METRICS_DIR),
]
subprocess.run(benchmark_cmd, check=True)

## 8. Save results to Drive + summary

In [ ]:
import json
import shutil

shutil.copytree(CALIBRATION_DIR, DRIVE_RESULTS_DIR / 'calibration', dirs_exist_ok=True)
shutil.copytree(BENCHMARK_METRICS_DIR, DRIVE_RESULTS_DIR / 'metrics', dirs_exist_ok=True)

results_path = BENCHMARK_METRICS_DIR / 'codec_benchmark.json'
results = json.loads(results_path.read_text(encoding='utf-8'))
checkpoint_epoch = results['checkpoint_epoch']
frames_evaluated = results['frames_evaluated']

sep = '=' * 70
print(sep)
print(f'Codec benchmark on {DATASET_LABEL!r} - checkpoint epoch {checkpoint_epoch} - {frames_evaluated} test frames')
print(sep)
for row in results['configurations']:
    bits = row['bits']
    mode = row['mode']
    psnr_db = row['psnr_db']
    bpp = row['mean_total_bpp']
    ratio = row['mean_compression_ratio']
    print(f'{bits}-bit ({mode}): {psnr_db:.3f} dB, {bpp:.4f} BPP, {ratio:.2f}x vs raw RGB')
print(sep)
print()
print(f'All results saved to: {DRIVE_RESULTS_DIR}')
print('  reconstruction.png   - visual Original | Reconstruction comparison')
print('  calibration/         - one calibration file per bit width')
print('  metrics/codec_benchmark.json / .csv - full measured benchmark')

## Testing a different dataset next time

In the config cell at the top: set `USE_BUILTIN_DAVIS = False`, then set `DATASET_LABEL` and `DATASET_INPUT_PATH` to point at your new dataset. Runtime -> Run all. Everything downstream (ingestion, reconstruction, calibration, benchmarking, saving to Drive) is dataset-agnostic and will run against the new dataset automatically, writing results to a differently-named Drive folder so nothing from previous runs gets overwritten.

If your next dataset is hosted on Kaggle rather than sitting in Drive already, reuse the Kaggle-download pattern from `colab_train_vimeo.ipynb` (upload `kaggle.json` once, `kaggle datasets download -d <owner>/<slug>`) to pull it into a local folder first, then point `DATASET_INPUT_PATH` at that folder - the same way section 4 above downloads DAVIS automatically.